# 06 — Transparent multimodal fusion

This notebook exercises the preregistered **late-fusion** and **visual-candidate reranking** machinery on tiny fixtures. It is a software/research-protocol smoke test, not a claim that graph information already improves the real heritage benchmark.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
from caypollard.embeddings.store import EmbeddingTable
from caypollard.fusion import (
    SimilarityBounds,
    late_fusion_embedding_table,
    rerank_visual_candidates,
)

In [ ]:
ids = ("a", "b", "c", "d")
visual = EmbeddingTable(
    ids=ids,
    vectors=np.asarray([[1, 0], [0.9, 0.1], [0.1, 0.9], [0, 1]], dtype=np.float32),
    metadata={"method": "visual-fixture"},
)
graph = EmbeddingTable(
    ids=ids,
    vectors=np.asarray([[1, 0], [0.1, 0.9], [0.95, 0.05], [0, 1]], dtype=np.float32),
    metadata={"method": "graph-fixture", "projection": "G1-fixture"},
)
vb = SimilarityBounds(low=-1.0, high=1.0)
gb = SimilarityBounds(low=-1.0, high=1.0)

## Weighted late fusion

With fixed validation-derived affine bounds, weighted similarity ranking can be represented exactly as a concatenated cosine space.

In [ ]:
fused = late_fusion_embedding_table(visual, graph, alpha=0.5, visual_bounds=vb, graph_bounds=gb)
fused.metadata

In [ ]:
query_index = fused.ids.index("a")
order = np.argsort(-(fused.vectors @ fused.vectors[query_index]))
[fused.ids[i] for i in order if i != query_index]

## Visual retrieval followed by graph reranking

This keeps the visual candidate pool explicit. It is useful when the desired product behaviour is “find visually plausible candidates, then inject historical context,” rather than globally replacing visual similarity.

In [ ]:
visual_only = rerank_visual_candidates(
    visual, graph, query_id="a", candidate_ids=ids, candidate_k=3,
    alpha=1.0, visual_bounds=vb, graph_bounds=gb,
)
graph_heavy = rerank_visual_candidates(
    visual, graph, query_id="a", candidate_ids=ids, candidate_k=3,
    alpha=0.1, visual_bounds=vb, graph_bounds=gb,
)
visual_only, graph_heavy

## Interpretation constraint

The full experiment selects `alpha` on validation nDCG@10 and evaluates the chosen value once on test data. A gain involving `G0` remains an oracle/control result; only `G1` or `G2` is eligible for the headline claim.